# Define an area and analyse the spatial distribution of HOSTRADA climate variables

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from pyproj import Transformer
from hostrada4py import hostradaArea as ha
from hostrada4py import hostradaCities as hs
from hostrada4py import hostradaRegions as hr
import os
import ipywidgets as widgets
from IPython.display import display

from hostrada4py import hostradaAreaUI as haui
from hostrada4py import hostradaAreaMapUI as hamui

os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"


## Selection of the HOSTRADA value

Select the desired climate variable with the dropdown. The selected value is written directly to `HOSTRADA_VAR` and is therefore used by all subsequent cells.

In [ ]:
# Climate variables available in the original notebook.
CLIMATE_VARIABLES = {
    "Outside air temperature (tas)": "tas",
    "Wind speed (sfcWind)": "sfcWind",
    "Wind direction (sfcWind_direction)": "sfcWind_direction",
    "Urban Heat Island Intensity (uhi)": "uhi",
    "Global radiation (rsds)": "rsds",
    "Cloud cover (clt)": "clt",
    "Relative humidity (hurs)": "hurs",
    "Water vapor mixing ratio (mixr)": "mixr",
    "Dew point temperature (tdew)": "tdew",
}

HOSTRADA_VAR = "tas"

climate_dropdown = widgets.Dropdown(
    options=[(label, value) for label, value in CLIMATE_VARIABLES.items()],
    value=HOSTRADA_VAR,
    description="Climate variable:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)
climate_status = widgets.HTML()


def _set_climate_variable(change=None):
    global HOSTRADA_VAR
    HOSTRADA_VAR = climate_dropdown.value
    selected_label = next(
        label for label, value in CLIMATE_VARIABLES.items() if value == HOSTRADA_VAR
    )
    climate_status.value = (
        f"<b>Selected:</b> {selected_label} &nbsp; "
        f"(<code>HOSTRADA_VAR = '{HOSTRADA_VAR}'</code>)"
    )


climate_dropdown.observe(_set_climate_variable, names="value")
_set_climate_variable()
display(widgets.VBox([climate_dropdown, climate_status]))


## Definition of the area and the time period of the HOSTRADA values

Choose one of the predefined city or region polygons from the dropdown. Alternatively choose **Custom polygon – click points on map** and define the polygon on the Leaflet map. The active geometry is written directly to `polygon_points` in the original `(longitude, latitude)` format.

Set the download period with the date and hour selectors below. The selected UTC boundaries are written directly to `start_utc` and `end_utc` and are therefore used unchanged by the download cell.

In [ ]:
hostrada_area_ui = haui.HostradaAreaUI(
    notebook_namespace=globals(),
    cities_module=hs,
    regions_module=hr,
    initial_area_name="City – Berlin",
    initial_start_utc="2025-01-01T00:00:00",
    initial_end_utc="2025-01-31T23:00:00",
)
hostrada_area_ui.show()


## Download of the HOSTRADA values

In [ ]:
if not polygon_points:
    raise ValueError(
        "No valid area is selected. Choose a predefined area or finish drawing "
        "a custom polygon on the Leaflet map before running this cell."
    )

gdf = ha.extract_values_for_polygon(
    var= HOSTRADA_VAR,
    polygon_lonlat=polygon_points,
    start_utc=start_utc,
    end_utc=end_utc,
    selection_mode="within",
    return_geodataframe=True)
print(f"Number of data records: {len(gdf)}")

summary = ha.summarize_values_period(gdf, var = HOSTRADA_VAR)

gdf.to_file("HOSTRADA_poly_" + HOSTRADA_VAR + ".geojson", driver="GeoJSON")
gdf.drop(columns="geometry").to_csv("HOSTRADA_poly_" + HOSTRADA_VAR + ".csv", index=False)
summary.to_csv("HOSTRADA_poly_summary_" + HOSTRADA_VAR + ".csv", index=False)
print("\nResults stored:")
print(" - HOSTRADA_poly_" + HOSTRADA_VAR + ".geojson")
print(" - HOSTRADA_poly_" + HOSTRADA_VAR + ".csv")
print(" - HOSTRADA_poly_summary_" + HOSTRADA_VAR + ".csv")

## Spatial distribution of the HOSTRADA variable for one selected time period

Select a day and one of the actually available UTC times from the downloaded data. Click **Update climate map** to generate the two-dimensional map for that time point.

In [ ]:
hostrada_area_map_ui = hamui.HostradaAreaMapUI(
    notebook_namespace=globals(),
    hostrada_area_module=ha,
    original_default_time="2025-01-08T12:00:00",
    output_directory="./html",
)
hostrada_area_map_ui.show()


## Hourly time series of area mean values of the whole polygon.

In [ ]:
if HOSTRADA_VAR == "tas":
    title = "Air temperature in °C"
elif HOSTRADA_VAR == "uhi":
    title = "Urban Heat Island Intensity in °C"
elif HOSTRADA_VAR == "sfcWind":
    title = "Wind speed in m/s"
elif HOSTRADA_VAR == "sfcWind_direction":
    title = "Wind direction in degree"
elif HOSTRADA_VAR == "rsds":
    title = "Global radiation in W/m2"
elif HOSTRADA_VAR == "clt":
    title = "Cloud cover in eighth"
elif HOSTRADA_VAR == "hurs":
    title = "Relative humidity in percent"
elif HOSTRADA_VAR == "tdew":
    title = "Dew point temperature in °C"
elif HOSTRADA_VAR == "mixr":
    title = "Water vapor mixing ratio in g H20/kg dry air"
else:
    title = "unknown"

summary["time"] = pd.to_datetime(summary["time"])

plt.figure(figsize=(12, 5))
plt.plot(summary["time"], summary["value_mean"], label="Mittelwert")
plt.fill_between(summary["time"], summary["value_min"], summary["value_max"], alpha=0.2, label="Min–Max")
plt.xlabel("time")
#plt.ylabel(title)
plt.title(title)
plt.grid(True)
plt.legend()
plt.show()
